<a href="https://colab.research.google.com/github/ravichandranpapitha-sys/Final-Year-Project-Papitha1/blob/main/01_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Student Performance Prediction Using Machine Learning**
### **Exploratory Data Analysis (EDA)**

**Module:** 7PAM2002 Data Science Project  
**Student:** Papitha Ravichandran (24078677)  
**University of Hertfordshire — MSc Data Science**  
**Dataset:** Open University Learning Analytics Dataset (OULAD)


## **Step 1 — Import Libraries**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully.")

Libraries loaded successfully.


## **Step 2 — Load All OULAD Tables**

In [4]:
DATA_DIR = 'dataset/'

courses       = pd.read_csv(DATA_DIR + 'courses.csv')
assessments   = pd.read_csv(DATA_DIR + 'assessments.csv')
student_info  = pd.read_csv(DATA_DIR + 'studentInfo.csv')
student_reg   = pd.read_csv(DATA_DIR + 'studentRegistration.csv')
student_assess = pd.read_csv(DATA_DIR + 'studentAssessment.csv')
vle           = pd.read_csv(DATA_DIR + 'vle.csv')

tables = {
    'courses': courses, 'assessments': assessments,
    'studentInfo': student_info, 'studentRegistration': student_reg,
    'studentAssessment': student_assess, 'vle': vle
}

print("Table shapes:")
for name, df in tables.items():
    print(f"  {name:25s} {str(df.shape):>15s}  cols: {df.columns.tolist()}")

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/courses.csv'

## **Step 3 — Data Overview and Schema**

The OULAD consists of 7 relational tables linked by `code_module`, `code_presentation`, `id_student`, `id_assessment`, and `id_site`. The central table is `studentInfo` which contains the target variable `final_result` with four classes: Distinction, Pass, Fail, and Withdrawn.

In [ ]:
print("=== studentInfo — first 5 rows ===")
print(student_info.head().to_string())
print("\n=== Data types ===")
print(student_info.dtypes)
print("\n=== Basic statistics (numeric columns) ===")
print(student_info.describe().to_string())

## **Step 4 — Missing Value Analysis**

In [ ]:
print("=== Missing values per table ===\n")
for name, df in tables.items():
    total = df.isnull().sum()
    missing = total[total > 0]
    if len(missing) > 0:
        pct = (missing / len(df) * 100).round(1)
        info = pd.DataFrame({'missing_count': missing, 'pct': pct})
        print(f"{name} ({len(df)} rows):")
        print(info.to_string())
        print()
    else:
        print(f"{name}: no missing values\n")

# Specific note on imd_band
print(f"\nimd_band has {student_info['imd_band'].isnull().sum()} missing values ({student_info['imd_band'].isnull().mean()*100:.1f}%)")
print("These likely represent students from outside England where IMD is not applicable.")
print("\ndate_unregistration is NaN for students who did NOT withdraw — this is expected, not an error.")

## **Step 5 — Target Variable Distribution (final_result)**

This is the most critical step for RQ1. Understanding the class distribution reveals the imbalance that will drive our modelling strategy.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
order = ['Distinction', 'Pass', 'Fail', 'Withdrawn']
colors = ['#2ecc71', '#3498db', '#e74c3c', '#95a5a6']
counts = student_info['final_result'].value_counts().reindex(order)
axes[0].bar(order, counts.values, color=colors, edgecolor='black', linewidth=0.5)
for i, (v, c) in enumerate(zip(counts.values, counts.values)):
    axes[0].text(i, v + 200, f'{c:,}\n({c/len(student_info)*100:.1f}%)', ha='center', fontsize=10)
axes[0].set_title('Distribution of Final Result (Target Variable)', fontweight='bold')
axes[0].set_ylabel('Number of Students')

# Pie chart
axes[1].pie(counts.values, labels=order, colors=colors, autopct='%1.1f%%',
            startangle=90, pctdistance=0.85, wedgeprops=dict(linewidth=1, edgecolor='white'))
axes[1].set_title('Proportion of Each Outcome', fontweight='bold')

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey finding:")
print(f"  Pass: {counts['Pass']:,} ({counts['Pass']/len(student_info)*100:.1f}%)")
print(f"  Withdrawn: {counts['Withdrawn']:,} ({counts['Withdrawn']/len(student_info)*100:.1f}%)")
print(f"  Fail: {counts['Fail']:,} ({counts['Fail']/len(student_info)*100:.1f}%)")
print(f"  Distinction: {counts['Distinction']:,} ({counts['Distinction']/len(student_info)*100:.1f}%)")
print(f"\n  Pass-to-Distinction ratio: {counts['Pass']/counts['Distinction']:.1f}:1")
print(f"  (Pass+Withdrawn) vs (Fail+Distinction) = {(counts['Pass']+counts['Withdrawn'])/len(student_info)*100:.1f}% vs {(counts['Fail']+counts['Distinction'])/len(student_info)*100:.1f}%")
print("\nThe dataset is imbalanced — Distinction is the minority class at 9.3%.")
print("SMOTE/ADASYN will be needed to address this for RQ1.")

## **Step 6 — Demographic Analysis**

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Gender
gender_outcome = pd.crosstab(student_info['gender'], student_info['final_result'], normalize='index').reindex(columns=order) * 100
gender_outcome.plot(kind='bar', stacked=True, color=colors, ax=axes[0, 0], edgecolor='black', linewidth=0.5)
axes[0, 0].set_title('Outcome by Gender', fontweight='bold')
axes[0, 0].set_ylabel('Percentage')
axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=0)
axes[0, 0].legend(title='Result', bbox_to_anchor=(1.0, 1.0))

# Age band
age_outcome = pd.crosstab(student_info['age_band'], student_info['final_result'], normalize='index').reindex(columns=order) * 100
age_order = ['0-35', '35-55', '55<=']
age_outcome = age_outcome.reindex(age_order)
age_outcome.plot(kind='bar', stacked=True, color=colors, ax=axes[0, 1], edgecolor='black', linewidth=0.5)
axes[0, 1].set_title('Outcome by Age Band', fontweight='bold')
axes[0, 1].set_ylabel('Percentage')
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=0)
axes[0, 1].legend(title='Result', bbox_to_anchor=(1.0, 1.0))

# Highest education
edu_outcome = pd.crosstab(student_info['highest_education'], student_info['final_result'], normalize='index').reindex(columns=order) * 100
edu_order = ['No Formal quals', 'Lower Than A Level', 'A Level or Equivalent', 'HE Qualification', 'Post Graduate Qualification']
edu_outcome = edu_outcome.reindex(edu_order)
edu_outcome.plot(kind='barh', stacked=True, color=colors, ax=axes[1, 0], edgecolor='black', linewidth=0.5)
axes[1, 0].set_title('Outcome by Highest Education', fontweight='bold')
axes[1, 0].set_xlabel('Percentage')
axes[1, 0].legend(title='Result', bbox_to_anchor=(1.0, 1.0))

# Disability
dis_outcome = pd.crosstab(student_info['disability'], student_info['final_result'], normalize='index').reindex(columns=order) * 100
dis_outcome.plot(kind='bar', stacked=True, color=colors, ax=axes[1, 1], edgecolor='black', linewidth=0.5)
axes[1, 1].set_title('Outcome by Disability Status', fontweight='bold')
axes[1, 1].set_ylabel('Percentage')
axes[1, 1].set_xticklabels(['No Disability', 'Has Disability'], rotation=0)
axes[1, 1].legend(title='Result', bbox_to_anchor=(1.0, 1.0))

plt.tight_layout()
plt.savefig('demographics_vs_outcome.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key demographic findings:")
print("  - Males have slightly higher withdrawal/fail rates than females.")
print("  - Younger students (0-35) form the majority; older students (55+) have higher pass rates.")
print("  - Students with Post Graduate qualifications have the highest Distinction rate.")
print("  - Students with No Formal qualifications have the highest Fail/Withdrawn rate.")

## **Step 7 — Deprivation Index (IMD Band) Analysis**

In [ ]:
imd_data = student_info.dropna(subset=['imd_band']).copy()

# Fix inconsistent labelling
imd_data['imd_band'] = imd_data['imd_band'].replace({'10-20': '10-20%'})

imd_order = ['0-10%', '10-20%', '20-30%', '30-40%', '40-50%', '50-60%', '60-70%', '70-80%', '80-90%', '90-100%']
imd_outcome = pd.crosstab(imd_data['imd_band'], imd_data['final_result'], normalize='index').reindex(columns=order) * 100
imd_outcome = imd_outcome.reindex(imd_order)

fig, ax = plt.subplots(figsize=(14, 6))
imd_outcome.plot(kind='bar', stacked=True, color=colors, ax=ax, edgecolor='black', linewidth=0.5)
ax.set_title('Outcome by IMD Band (Index of Multiple Deprivation)', fontweight='bold')
ax.set_ylabel('Percentage')
ax.set_xlabel('IMD Band (0-10% = Most Deprived, 90-100% = Least Deprived)')
ax.legend(title='Result', bbox_to_anchor=(1.02, 1.0))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('imd_vs_outcome.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key finding: Students from more deprived areas (lower IMD) tend to have")
print("higher Fail/Withdrawn rates, suggesting socioeconomic background is a predictor.")

## **Step 8 — Module and Presentation Analysis (RQ3 relevance)**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# By module
mod_outcome = pd.crosstab(student_info['code_module'], student_info['final_result'], normalize='index').reindex(columns=order) * 100
mod_outcome.plot(kind='bar', stacked=True, color=colors, ax=axes[0], edgecolor='black', linewidth=0.5)
axes[0].set_title('Outcome by Module', fontweight='bold')
axes[0].set_ylabel('Percentage')
axes[0].legend(title='Result', bbox_to_anchor=(1.0, 1.0))
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# By presentation (temporal)
pres_outcome = pd.crosstab(student_info['code_presentation'], student_info['final_result'], normalize='index').reindex(columns=order) * 100
pres_order = ['2013B', '2013J', '2014B', '2014J']
pres_outcome = pres_outcome.reindex(pres_order)
pres_outcome.plot(kind='bar', stacked=True, color=colors, ax=axes[1], edgecolor='black', linewidth=0.5)
axes[1].set_title('Outcome by Presentation (Temporal View for RQ3)', fontweight='bold')
axes[1].set_ylabel('Percentage')
axes[1].legend(title='Result', bbox_to_anchor=(1.0, 1.0))
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('module_presentation_outcome.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key findings:")
print("  - Module difficulty varies substantially (e.g., BBB, DDD have higher fail rates).")
print("  - Outcome distributions are relatively stable across presentations (2013B-2014J),")
print("    which is promising for RQ3 — models may generalise across semesters.")

# Student counts per presentation
print("\nStudents per presentation:")
print(student_info['code_presentation'].value_counts().sort_index().to_string())

## **Step 9 — Assessment Score Analysis**

In [ ]:
# Merge assessment scores with student info
assess_merged = student_assess.merge(
    assessments[['id_assessment', 'code_module', 'code_presentation', 'assessment_type', 'weight']],
    on='id_assessment', how='left'
)
assess_merged = assess_merged.merge(
    student_info[['code_module', 'code_presentation', 'id_student', 'final_result']],
    on=['code_module', 'code_presentation', 'id_student'], how='left'
)

print(f"Merged assessment records: {len(assess_merged):,}")
print(f"Assessment types: {assess_merged['assessment_type'].value_counts().to_dict()}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Score distribution by outcome
for result, color in zip(order, colors):
    subset = assess_merged[assess_merged['final_result'] == result]['score'].dropna()
    axes[0].hist(subset, bins=50, alpha=0.5, label=result, color=color, density=True)
axes[0].set_title('Assessment Score Distribution by Outcome', fontweight='bold')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Density')
axes[0].legend()

# Average score by outcome
avg_score = assess_merged.groupby('final_result')['score'].mean().reindex(order)
axes[1].bar(order, avg_score.values, color=colors, edgecolor='black', linewidth=0.5)
for i, v in enumerate(avg_score.values):
    axes[1].text(i, v + 0.5, f'{v:.1f}', ha='center', fontweight='bold')
axes[1].set_title('Mean Assessment Score by Outcome', fontweight='bold')
axes[1].set_ylabel('Mean Score')

plt.tight_layout()
plt.savefig('assessment_scores.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nMean scores: {avg_score.to_dict()}")
print("\nDistinction students score ~83 on average vs ~55 for Fail students.")
print("Assessment scores will be a strong predictor feature for RQ2.")

## **Step 10 — VLE Engagement Analysis**

The `studentVle.csv` file has 10.6 million rows. Rather than loading it entirely into memory, we aggregate clicks per student on-the-fly using chunked reading.

In [ ]:
# Aggregate VLE clicks per student using chunked reading
print("Aggregating VLE clicks per student (chunked reading of 10.6M rows)...")

click_agg = pd.DataFrame()
chunk_iter = pd.read_csv(DATA_DIR + 'studentVle.csv', chunksize=500000)

for i, chunk in enumerate(chunk_iter):
    agg = chunk.groupby(['code_module', 'code_presentation', 'id_student'])['sum_click'].sum().reset_index()
    click_agg = pd.concat([click_agg, agg], ignore_index=True)
    if (i+1) % 5 == 0:
        print(f"  Processed {(i+1)*500000:,} rows...")

# Final aggregation (sum across chunks)
vle_student = click_agg.groupby(['code_module', 'code_presentation', 'id_student'])['sum_click'].sum().reset_index()
vle_student.rename(columns={'sum_click': 'total_clicks'}, inplace=True)

print(f"\nVLE summary: {len(vle_student):,} student-module records")
print(f"Total clicks range: {vle_student['total_clicks'].min()} - {vle_student['total_clicks'].max():,}")
print(f"Mean clicks: {vle_student['total_clicks'].mean():.0f}, Median: {vle_student['total_clicks'].median():.0f}")

## **Step 11 — VLE Engagement vs Outcome**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Merge with student info
vle_outcome = vle_student.merge(
    student_info[['code_module', 'code_presentation', 'id_student', 'final_result']],
    on=['code_module', 'code_presentation', 'id_student'], how='inner'
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot
vle_outcome.boxplot(column='total_clicks', by='final_result', ax=axes[0],
                     positions=[order.index(x) for x in vle_outcome['final_result'].unique()
                                if x in order])
# Cleaner boxplot
axes[0].cla()
bp_data = [vle_outcome[vle_outcome['final_result'] == r]['total_clicks'].clip(upper=vle_outcome['total_clicks'].quantile(0.95)) for r in order]
bp = axes[0].boxplot(bp_data, labels=order, patch_artist=True, showfliers=False)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_title('Total VLE Clicks by Outcome (capped at 95th percentile)', fontweight='bold')
axes[0].set_ylabel('Total Clicks')

# Mean clicks bar
mean_clicks = vle_outcome.groupby('final_result')['total_clicks'].mean().reindex(order)
axes[1].bar(order, mean_clicks.values, color=colors, edgecolor='black', linewidth=0.5)
for i, v in enumerate(mean_clicks.values):
    axes[1].text(i, v + 50, f'{v:,.0f}', ha='center', fontweight='bold')
axes[1].set_title('Mean VLE Clicks by Outcome', fontweight='bold')
axes[1].set_ylabel('Mean Total Clicks')

plt.suptitle('')
plt.tight_layout()
plt.savefig('vle_engagement.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key finding: Distinction students engage ~2-3x more with VLE than Withdrawn students.")
print("VLE engagement (total clicks) will be a critical feature for RQ1 and RQ2.")

## **Step 12 — VLE Activity Type Breakdown**

In [ ]:
# Aggregate by activity type per student
print("Aggregating VLE activity types per student...")

activity_agg = pd.DataFrame()
chunk_iter = pd.read_csv(DATA_DIR + 'studentVle.csv', chunksize=500000)

for chunk in chunk_iter:
    chunk_m = chunk.merge(vle[['id_site', 'activity_type']], on='id_site', how='left')
    agg = chunk_m.groupby(['code_module', 'code_presentation', 'id_student', 'activity_type'])['sum_click'].sum().reset_index()
    activity_agg = pd.concat([activity_agg, agg], ignore_index=True)

activity_student = activity_agg.groupby(['code_module', 'code_presentation', 'id_student', 'activity_type'])['sum_click'].sum().reset_index()

# Pivot to get one column per activity type
activity_pivot = activity_student.pivot_table(
    index=['code_module', 'code_presentation', 'id_student'],
    columns='activity_type', values='sum_click', fill_value=0
).reset_index()

# Top activity types
total_by_type = activity_student.groupby('activity_type')['sum_click'].sum().sort_values(ascending=False)
print("\nTop 10 VLE activity types by total clicks:")
print(total_by_type.head(10).to_string())

fig, ax = plt.subplots(figsize=(14, 6))
top10 = total_by_type.head(10)
ax.barh(top10.index[::-1], top10.values[::-1], color=sns.color_palette('viridis', 10)[::-1], edgecolor='black', linewidth=0.5)
ax.set_title('Top 10 VLE Activity Types by Total Clicks', fontweight='bold')
ax.set_xlabel('Total Clicks (millions)')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.tight_layout()
plt.savefig('vle_activity_types.png', dpi=150, bbox_inches='tight')
plt.show()

## **Step 13 — Registration Timing Analysis**

In [ ]:
reg_outcome = student_reg.merge(
    student_info[['code_module', 'code_presentation', 'id_student', 'final_result']],
    on=['code_module', 'code_presentation', 'id_student'], how='inner'
)

fig, ax = plt.subplots(figsize=(14, 6))
for result, color in zip(order, colors):
    subset = reg_outcome[reg_outcome['final_result'] == result]['date_registration'].dropna()
    ax.hist(subset, bins=80, alpha=0.5, label=result, color=color, density=True)

ax.set_title('Registration Date Distribution by Outcome', fontweight='bold')
ax.set_xlabel('Days Before Module Start (negative = early)')
ax.set_ylabel('Density')
ax.axvline(x=0, color='black', linestyle='--', alpha=0.5, label='Module start')
ax.legend()
plt.tight_layout()
plt.savefig('registration_timing.png', dpi=150, bbox_inches='tight')
plt.show()

# Mean registration date by outcome
mean_reg = reg_outcome.groupby('final_result')['date_registration'].mean().reindex(order)
print("Mean registration date (days before start) by outcome:")
for r, v in mean_reg.items():
    print(f"  {r}: {v:.0f} days")
print("\nStudents who register earlier tend to perform better.")

## **Step 14 — Studied Credits and Previous Attempts**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Studied credits by outcome
for result, color in zip(order, colors):
    subset = student_info[student_info['final_result'] == result]['studied_credits']
    axes[0].hist(subset, bins=30, alpha=0.5, label=result, color=color, density=True)
axes[0].set_title('Studied Credits Distribution by Outcome', fontweight='bold')
axes[0].set_xlabel('Studied Credits')
axes[0].set_ylabel('Density')
axes[0].legend()

# Previous attempts
prev_outcome = pd.crosstab(
    student_info['num_of_prev_attempts'].clip(upper=3).replace({3: '3+'}),
    student_info['final_result'], normalize='index'
).reindex(columns=order) * 100
prev_outcome.plot(kind='bar', stacked=True, color=colors, ax=axes[1], edgecolor='black', linewidth=0.5)
axes[1].set_title('Outcome by Number of Previous Attempts', fontweight='bold')
axes[1].set_ylabel('Percentage')
axes[1].set_xlabel('Previous Attempts')
axes[1].legend(title='Result', bbox_to_anchor=(1.0, 1.0))
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('credits_attempts.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key findings:")
print("  - Students taking more credits tend to have diverse outcomes.")
print(f"  - {(student_info['num_of_prev_attempts']==0).mean()*100:.1f}% of students are first-time takers.")
print("  - Repeating students may have different outcome patterns.")

## **Step 15 — Feature Correlation Analysis**

In [ ]:
# Build a merged feature set for correlation
merged = student_info.merge(vle_student, on=['code_module', 'code_presentation', 'id_student'], how='left')

# Add mean assessment score
mean_score = assess_merged.groupby(['code_module', 'code_presentation', 'id_student'])['score'].mean().reset_index()
mean_score.rename(columns={'score': 'mean_score'}, inplace=True)
merged = merged.merge(mean_score, on=['code_module', 'code_presentation', 'id_student'], how='left')

# Add registration date
merged = merged.merge(
    student_reg[['code_module', 'code_presentation', 'id_student', 'date_registration']],
    on=['code_module', 'code_presentation', 'id_student'], how='left'
)

# Encode target numerically for correlation
result_map = {'Distinction': 3, 'Pass': 2, 'Fail': 1, 'Withdrawn': 0}
merged['result_numeric'] = merged['final_result'].map(result_map)

# Select numeric features
num_cols = ['num_of_prev_attempts', 'studied_credits', 'total_clicks', 'mean_score',
            'date_registration', 'result_numeric']
corr = merged[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=1, ax=ax,
            xticklabels=['Prev Attempts', 'Credits', 'VLE Clicks', 'Mean Score', 'Reg Date', 'Result'],
            yticklabels=['Prev Attempts', 'Credits', 'VLE Clicks', 'Mean Score', 'Reg Date', 'Result'])
ax.set_title('Correlation Matrix of Key Numeric Features', fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key correlations with result_numeric (higher = better outcome):")
result_corr = corr['result_numeric'].drop('result_numeric').sort_values(ascending=False)
for feat, val in result_corr.items():
    print(f"  {feat}: {val:+.3f}")

## **Step 16 — Temporal Stability Check (RQ3)**

For RQ3, we need to verify that outcome distributions and feature relationships are comparable across presentations (semesters), so that training on earlier data and testing on later data is valid.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

presentations = ['2013B', '2013J', '2014B', '2014J']
for i, pres in enumerate(presentations):
    ax = axes[i // 2][i % 2]
    subset = student_info[student_info['code_presentation'] == pres]
    outcome_pct = subset['final_result'].value_counts(normalize=True).reindex(order) * 100
    ax.bar(order, outcome_pct.values, color=colors, edgecolor='black', linewidth=0.5)
    for j, v in enumerate(outcome_pct.values):
        ax.text(j, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9)
    ax.set_title(f'Presentation {pres} (n={len(subset):,})', fontweight='bold')
    ax.set_ylabel('Percentage')
    ax.set_ylim(0, 50)

plt.suptitle('Outcome Distribution Across Presentations (Temporal View)', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('temporal_stability.png', dpi=150, bbox_inches='tight')
plt.show()

# Compute stability metrics
print("Outcome percentages per presentation:")
stability = pd.crosstab(student_info['code_presentation'], student_info['final_result'], normalize='index').reindex(columns=order) * 100
print(stability.round(1).to_string())
print(f"\nStd deviation across presentations:")
print(stability.std().round(2).to_string())
print("\nLow std (<3%) indicates stable distributions — good for RQ3 temporal splits.")

## **Step 17 — EDA Summary and Key Findings**

### Target Variable (RQ1)
- Four outcome classes: Pass (37.9%), Withdrawn (31.2%), Fail (21.6%), Distinction (9.3%)
- Significant class imbalance — Distinction is the smallest class at 9.3%
- SMOTE/ADASYN resampling will be essential for minority-class prediction

### Demographic Features (RQ2)
- Gender, age band, highest education, IMD band, and disability all show association with outcomes
- Students from deprived areas and those with lower prior education have higher fail/withdrawal rates
- Post-graduate qualification holders have the best outcomes

### Behavioural Features (RQ2)
- VLE engagement (total clicks) is strongly correlated with positive outcomes
- Distinction students engage 2-3x more than Withdrawn students
- Early registration is associated with better outcomes
- Assessment scores are the strongest single predictor of final result

### Temporal Stability (RQ3)
- Outcome distributions are remarkably stable across all 4 presentations (2013B-2014J)
- Standard deviation of outcome percentages is under 3% — excellent for temporal validation
- This supports the feasibility of training on 2013 data and testing on 2014 data

### Features for Modelling
The following features will be engineered for the modelling phase:
- **Demographic:** gender, age_band, highest_education, imd_band, disability, region
- **Academic:** studied_credits, num_of_prev_attempts, mean_score, score_std, num_assessments_submitted
- **Behavioural:** total_clicks, clicks per activity type (oucontent, forumng, resource, etc.)
- **Temporal:** date_registration, early_engagement (clicks in first 25% of module)